In [52]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [53]:
df = pd.read_csv("/content/features_rahim.csv")

In [54]:
# Extract only numeric values from rating_count
df["rating_count"] = df["rating_count"].apply(lambda x: re.sub(r"\D", "", str(x)) if pd.notnull(x) else x)
df["rating_count"] = pd.to_numeric(df["rating_count"], errors="coerce")

In [55]:
# Extract only numeric values from ratings
df["rating"] = df["rating"].apply(lambda x: re.sub(r"\D", "", str(x)) if pd.notnull(x) else x)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

In [56]:
# Selecting relevant features
features = ["actual_price", "discounted_price", "discount_percentage", "rating_count", "avg_price_category", "price_difference_category", "rating"]
df = df.dropna(subset=features)

In [57]:
X = df[features]

In [58]:
# Creating Discount Price Category
bins = [-1, 0, 10, 30, 100]  # Define bin edges for discount percentage
labels = ["No Discount", "Low", "Medium", "High"]  # Corresponding labels
df["discount_price_category"] = pd.cut(df["discount_percentage"], bins=bins, labels=labels)

In [59]:
# Encoding categorical target variable
y = df["discount_price_category"].astype(str)  # Ensure categorical format
le = LabelEncoder()
y = le.fit_transform(y)

In [60]:
# Splitting the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [61]:
# Model 1: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest Precision:", precision_score(y_test, y_pred_rf, average='weighted'))
print("Random Forest Recall:", recall_score(y_test, y_pred_rf, average='weighted'))
print("Random Forest F1 Score:", f1_score(y_test, y_pred_rf, average='weighted'))

Random Forest Accuracy: 0.9977311401020987
Random Forest Precision: 0.998195631978614
Random Forest Recall: 0.9977311401020987
Random Forest F1 Score: 0.9975819747644019


In [62]:
# Model 2: Logistic Regression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("Logistic Regression Precision:", precision_score(y_test, y_pred_log, average='weighted'))
print("Logistic Regression Recall:", recall_score(y_test, y_pred_log, average='weighted'))
print("Logistic Regression F1 Score:", f1_score(y_test, y_pred_log, average='weighted'))

Logistic Regression Accuracy: 0.998298355076574
Logistic Regression Precision: 0.9983630053854926
Logistic Regression Recall: 0.998298355076574
Logistic Regression F1 Score: 0.9982520174566967


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [63]:
nn_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
nn_model.fit(X_train, y_train)
y_pred_nn = nn_model.predict(X_test)
print("Neural Network Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Neural Network Precision:", precision_score(y_test, y_pred_nn, average='weighted'))
print("Neural Network Recall:", recall_score(y_test, y_pred_nn, average='weighted'))
print("Neural Network F1 Score:", f1_score(y_test, y_pred_nn, average='weighted'))

Neural Network Accuracy: 0.9784458309699376
Neural Network Precision: 0.9759080585288366
Neural Network Recall: 0.9784458309699376
Neural Network F1 Score: 0.9767435203293012


In [64]:
# Price Recommendation Function
def recommend_price(input_features, model):
    input_array = np.array(input_features).reshape(1, -1)
    recommended_category = model.predict(input_array)
    return le.inverse_transform(recommended_category)[0]

In [67]:
test_input = [500, 450, 10, 200, 480, 50, 4.5]  # Example feature values
print("Recommended discount category:", recommend_price(test_input, rf_model))

Recommended discount category: Medium


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
